In [ ]:
%pip install yfinance

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, mean_absolute_error

In [ ]:
# Last 6 months of data
df = yf.download("AAPL", period="6mo", interval="1d")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0] for col in df.columns]

df = df.reset_index()
df = df.sort_values("Date")

In [ ]:
# --- Feature Engineering ---

df["sma_10"] = df["Close"].rolling(10).mean()
df["sma_20"] = df["Close"].rolling(20).mean()

delta = df["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / avg_loss
df["rsi_14"] = 100 - (100 / (1 + rs))

ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()
df["macd"] = ema12 - ema26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

rolling_mean = df["Close"].rolling(20).mean()
rolling_std = df["Close"].rolling(20).std()
df["bb_upper"] = rolling_mean + (2 * rolling_std)
df["bb_lower"] = rolling_mean - (2 * rolling_std)
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["Close"]

df["return"] = df["Close"].pct_change()
df["return_3d"] = df["Close"].pct_change(3)
df["return_5d"] = df["Close"].pct_change(5)

df["volatility_5d"] = df["return"].rolling(5).std()
df["volatility_10d"] = df["return"].rolling(10).std()

df["dist_sma_10"] = (df["Close"] - df["sma_10"]) / df["sma_10"]
df["dist_sma_20"] = (df["Close"] - df["sma_20"]) / df["sma_20"]

# Targets
df["target_signal"] = (df["return"].shift(-1) > 0.002).astype(int)
df["target_price"]  = df["Close"].shift(-1)

df = df.dropna()

In [ ]:
feature_cols = [
    "rsi_14", "macd", "macd_signal", "bb_width",
    "return_3d", "return_5d", "volatility_5d", "volatility_10d",
    "dist_sma_10", "dist_sma_20", "Volume"
]

X = df[feature_cols]
y_signal = df["target_signal"]
y_price  = df["target_price"]

split_index = int(len(df) * 0.8)

X_train, X_test       = X[:split_index], X[split_index:]
y_sig_train, y_sig_test = y_signal[:split_index], y_signal[split_index:]
y_prc_train, y_prc_test = y_price[:split_index], y_price[split_index:]

In [ ]:
# --- Classifier: BUY / SELL / HOLD signal ---
rf_clf = RandomForestClassifier(n_estimators=500, max_depth=8, random_state=42)
rf_clf.fit(X_train, y_sig_train)

clf_pred = rf_clf.predict(X_test)
print("RF Classifier Accuracy:", accuracy_score(y_sig_test, clf_pred))
print(classification_report(y_sig_test, clf_pred))
print("ROC-AUC:", roc_auc_score(y_sig_test, rf_clf.predict_proba(X_test)[:, 1]))

In [ ]:
# --- Regressor: Next day closing price ---
rf_reg = RandomForestRegressor(n_estimators=500, max_depth=8, random_state=42)
rf_reg.fit(X_train, y_prc_train)

price_pred = rf_reg.predict(X_test)
print("RF Regressor MAE: $", round(mean_absolute_error(y_prc_test, price_pred), 2))

In [ ]:
# --- Combined output: signal + predicted price ---
results = X_test.copy()
results["actual_next_close"]    = y_prc_test.values
results["predicted_next_close"] = price_pred
results["actual_signal"]        = y_sig_test.values
results["predicted_signal"]     = clf_pred
print(results[["actual_next_close", "predicted_next_close", "actual_signal", "predicted_signal"]].tail(10))

In [ ]:
# --- Latest signal + next day price forecast ---
latest_features = X_test.iloc[[-1]]

prob           = rf_clf.predict_proba(latest_features)[:, 1][0]
predicted_price = rf_reg.predict(latest_features)[0]

if prob > 0.6:
    signal = "BUY"
elif prob < 0.4:
    signal = "SELL"
else:
    signal = "HOLD"

print(f"Signal:              {signal}")
print(f"Confidence:          {round(prob, 2)}")
print(f"Predicted Next Close: ${round(predicted_price, 2)}")